# 54 · Apply ruleset on NSSP's syndrome definitions, checked against ESSENCE's own query

## What this checks

**Apply ruleset** sorts records by rules written in Fieldwork. This check covers release 0.18.0, on four syndrome
definitions that the US National Syndromic Surveillance Program (NSSP) publishes: **ILI CCDD v1**, **CDC Broad Acute
Respiratory DD v1**, **Norovirus v1** and **CDC Measles CCDD v1**.

NSSP writes each definition as a query for ESSENCE, its surveillance system. Fieldwork cannot read that query, so each
definition was copied into Apply ruleset's own rules by hand, part by part. A slip in that copying would put visits in
the wrong syndrome, and nothing in Fieldwork would notice. So it is worth checking.

## How it is checked

We read NSSP's query as published, and run it ourselves, the way ESSENCE runs it. We do not read Fieldwork's copy.

The rules for reading the query come from CDC's guide, *Free-text Coding in NSSP-ESSENCE*. A word is looked for anywhere
in the text, ignoring capitals. A word after "andnot" rules out only the word just before it. "And" joins before "or".
A bracket such as [018] stands for one character: 0, 1 or 8. Python's own pattern matching does the looking.

The data are the 24,328 made-up emergency department visits of Fieldwork's syndromic surveillance example: each visit's
chief complaint, and its discharge diagnosis in ICD-10, ICD-11 and SNOMED CT, as Fieldwork ran them.

We check two versions of the definitions. One is as NSSP publishes them, with ICD-10 and SNOMED CT codes. The other is
Fieldwork's own translation, with ICD-11 codes in place of ICD-10, which NSSP does not publish. For that one, we swap the
codes in NSSP's query using the same translation table, then run it.

## What should happen

1. **For each definition, the same visits.** The visits Fieldwork's rules flag are exactly the visits NSSP's query
   selects, with the ICD-10 and SNOMED CT codes.
2. **Only stated differences.** Fieldwork's copy says where it reads the query differently. In ILI, "sore" and
   "throat" must both be present, but not in that order, where ESSENCE needs the order. A visit that differs for that
   reason is counted, and does not fail the check. Any other difference does.
3. **The same for the ICD-11 version**, with the translation table applied.

These criteria were fixed before the check was run.

In [ ]:
# This check needs no package beyond Python's own.
print("No packages to load: the check uses Python's own pattern matching.")

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
from checks.check_54_nssp_syndromes import run

report = run(Path.cwd() / "fixtures" / "nssp-syndromes-001.json")
print("agrees:", report["agrees"], "over", report["visits"], "visits")
for version in ("published", "icd11"):
    for name, d in report[version].items():
        print(f"  {version:9} {name:18} ESSENCE {d['essence']:5}  Fieldwork {d['fieldwork']:5}  "
              f"{'same visits' if d['agrees'] else 'DIFFERENT'}")

## The result on record

Recorded on 26 November 2026, against release 0.18.0: **Agrees**.

Every definition selects the same visits both ways, with no stated difference needed:

- **ILI CCDD v1:** 2,177 visits.
- **CDC Broad Acute Respiratory DD v1:** 2,987 visits.
- **Norovirus v1:** 223 visits.
- **CDC Measles CCDD v1:** 10 visits.

The ICD-11 version selects the same visits too. On these visits it cannot differ: the only place the two code
systems part, a dyspnoea with a fever, is in no visit.

The check was shown able to fail. Read with the spaces around " flu " dropped, it finds 2,295 ILI visits and 327
Norovirus visits, and disagrees with Fieldwork.

Running the cell above repeats the check on the same data.

## What this does not show

In [ ]:
for item in report["notEstablished"]:
    print("-", item)